# 05 評価

`04_train` が保存したモデル（`models/cloud_base_v1.joblib`）を、**テストの年（2025）** で評価し、
ベースラインと比べて、成功基準（`docs/spec.md`「9. 分割と評価」）を判定する。結果は `reports/metrics.json` に保存する。

- モデルはここでは学習しない（読み込むだけ）。テストの年を見た後は、モデル・特徴量・成功基準を変えない。
- 比べる相手は `03_baseline` と同じ 3 つ：`rule_of_thumb`（概算式。正式なベースライン）、`always_cap`（いつも上限）、
  `rule_of_thumb_metar`（METAR の気温・露点で計算した概算式）。
- 成功基準（都市ごとに、両方）：
  1. 全体の MAE が、`always_cap` より 10% 以上小さい。
  2. 上限より下の行だけの MAE が、`rule_of_thumb` より 10% 以上小さい。
- 参考として、検証の年（2024）の指標も同じ形で並べる（検証の年は、木の数と学習期間を決めるのに使っているので、良く出やすい）。
- 既定は **サンプル**（`tests/fixtures/data_raw/` の 2 日ぶん。その場で小さく学習して、動くことだけを確かめる。数値に意味はない。一時的な場所に書く）。
  コミットしてある `reports/metrics.json` は、環境変数 `DATA_FETCH_MODE=full` を付けて全期間で実行したもの。外部とは通信しない。
- 入力は Open-Meteo の過去データ（再解析）。API が使うのは予報なので、実際の精度はここでの数値より悪い可能性がある（`docs/model-card.md`）。

出典：Weather data by Open-Meteo.com（CC BY 4.0）、Iowa Environmental Mesonet（Iowa State University）。

In [ ]:
import json
import os
import sys
import tempfile
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from app.cities import City
from ml import (
    archive,
    baseline,
    dataset,
    evaluate,
    features,
    metar,
    model,
    raw_store,
    split,
    target,
    train,
)

## 0. 設定・読み込み

In [ ]:
MODE = os.environ.get("DATA_FETCH_MODE", "sample")

if MODE == "full":
    OUTPUT = ROOT / "reports" / "metrics.json"
    SPLITS = {name: list(split.SPLIT_YEARS[name]) for name in (split.VALIDATION, split.TEST)}
    CRITERIA_SPLIT = split.TEST
elif MODE == "sample":
    OUTPUT = Path(tempfile.mkdtemp()) / "metrics.json"
    # fixture は 2025 年の 2 日ぶんだけなので、分割で絞らずに 1 つの組として評価する。
    SPLITS = {"sample": [2025]}
    CRITERIA_SPLIT = "sample"
else:
    raise ValueError("DATA_FETCH_MODE must be 'sample' or 'full'")

pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", "{:.2f}".format)
print(f"mode={MODE}", list(SPLITS))

In [ ]:
if MODE == "full":
    observations = dataset.load_dataset(ROOT / "data" / "processed" / "observations.parquet")
else:
    raw_root = ROOT / "tests" / "fixtures" / "data_raw"
    parts = {}
    for city in City:
        weather_path = raw_store.raw_path(raw_root, "open_meteo_archive", city, 2025, "json")
        metar_path = raw_store.raw_path(raw_root, "iem", city, 2025, "csv")
        parts[city.value] = (
            archive.parse_archive(weather_path.read_bytes().decode("utf-8")),
            metar.parse_metar(metar_path.read_bytes().decode("utf-8")),
        )
    observations = dataset.build_dataset(parts)

dataset.validate_dataset(observations)
X = features.features(observations)
y = target.target_m(observations)
observations["split"] = split.split(observations) if MODE == "full" else "sample"
print(observations.shape, observations["time"].min(), observations["time"].max())

In [ ]:
if MODE == "full":
    cloud_base = model.load(ROOT / "models" / "cloud_base_v1.joblib")
else:
    # `04_train` の出力は読まずに、同じ行でその場で学習する（動くことの確認だけ）。
    cloud_base = model.CloudBaseModel(
        booster=train.fit(X, y, X, y),
        train_years=(2025,),
        model_version=model.MODEL_VERSION_V1,
    )

print(cloud_base.model_version, cloud_base.train_years)

## 1. 予測

In [ ]:
baselines = {
    "rule_of_thumb": baseline.baseline_m(
        observations["temperature_c"], observations["dew_point_c"]
    ),
    "always_cap": pd.Series(target.CEILING_CAP_M, index=observations.index, dtype="float64"),
    "rule_of_thumb_metar": baseline.baseline_m(
        observations["metar_temperature_c"], observations["metar_dew_point_c"]
    ),
}


def report_for(rows, label, prediction):
    """`rows` の行の指標（全体・都市別・都市 × 季節別）。"""
    truth = y[rows]
    if label == "rule_of_thumb_metar":
        # 参考の予測だけ、予測が作れない行を評価から除く。
        truth = truth.where(prediction[rows].notna())
    return evaluate.report(observations[rows], truth, prediction[rows])


predictions = {"model": model.predict_m(cloud_base, X), **baselines}
pd.DataFrame(predictions).describe().T

## 2. 評価

全体・都市別・都市 × 季節別（季節は現地の月）。

In [ ]:
results = {}
for name, years in SPLITS.items():
    rows = observations["split"] == name
    results[name] = {
        "years": years,
        "predictions": {
            label: report_for(rows, label, prediction) for label, prediction in predictions.items()
        },
    }

by_city = pd.DataFrame(
    [
        {"split": name, "prediction": label, "city": city, **metrics}
        for name, result in results.items()
        for label, reports in result["predictions"].items()
        for city, metrics in reports["by_city"].items()
    ]
)
by_city

In [ ]:
by_season = pd.DataFrame(
    [
        {"split": name, "prediction": label, "city": city, "season": season, **metrics}
        for name, result in results.items()
        for label in ("model", "rule_of_thumb", "always_cap")
        for city, seasons in result["predictions"][label]["by_city_season"].items()
        for season, metrics in seasons.items()
    ]
)
by_season.pivot_table(
    index=["split", "city", "season"], columns="prediction", values=["mae_m", "mae_below_cap_m"]
)

## 3. 成功基準

テストの年で、都市ごとに判定する。判定は丸める前の値で行う（ちょうど 10% は満たす）。

In [ ]:
criteria_rows = observations["split"] == CRITERIA_SPLIT
success_criteria = {}
for city in sorted(observations["city"][criteria_rows].unique()):
    rows = criteria_rows & (observations["city"] == city)
    metrics = {
        label: evaluate.evaluate(y[rows], predictions[label][rows])
        for label in ("model", "always_cap", "rule_of_thumb")
    }
    result = evaluate.success_criteria(
        model=metrics["model"], always_cap=metrics["always_cap"], baseline=metrics["rule_of_thumb"]
    )
    success_criteria[city] = {
        key: round(value, 4) if isinstance(value, float) else value for key, value in result.items()
    }

pd.DataFrame(success_criteria).T

## 4. 保存

実行した時刻は入れない（再実行しても同じファイルになる）。

In [ ]:
report = {
    "mode": MODE,
    "model_version": cloud_base.model_version,
    "train_years": list(cloud_base.train_years),
    "cap_m": target.CEILING_CAP_M,
    "predictions": {
        "model": "LightGBM (models/cloud_base_v1.joblib), clipped to [0, cap_m]",
        "rule_of_thumb": "baseline: 125 * (temperature_c - dew_point_c) with Open-Meteo values",
        "always_cap": "reference: always cap_m",
        "rule_of_thumb_metar": "reference: formula with METAR temperature and dew point",
    },
    "success_criteria_split": CRITERIA_SPLIT,
    "success_criteria_min_improvement": evaluate.SUCCESS_MIN_IMPROVEMENT,
    "success_criteria": success_criteria,
    "splits": results,
}

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
text = json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + "\n"
OUTPUT.write_text(text, encoding="utf-8", newline="\n")
assert json.loads(OUTPUT.read_text(encoding="utf-8")) == report
print(OUTPUT.name, len(text))